# Topic: Secure File Handling - Directory traversal attacks, path normalization (pathlib), and prefix validation
 
## 1. THE DIRECTORY TRAVERSAL VULNERABILITY
- **Directory Traversal (Path Traversal)**: Occurs when user inputs are directly concatenated to construct file paths:
  `filepath = os.path.join("/var/www/uploads", user_filename)`
  - If an attacker passes a filename like `../../../../etc/passwd`, the path resolution traverses up the directory tree, escaping the intended folder sandbox to read sensitive system files.
 
## 2. MITIGATION: PATH RESOLUTION & PREFIX CHECK
- **Path Normalization**:
  - Python's `pathlib.Path.resolve()` or `os.path.abspath()` fully resolves all symbolic links, relative references, and `../` dots, returning the absolute, canonical file path.
- **Prefix Verification**:
  - After resolving the absolute path, verify that it begins with the absolute path of the designated sandbox directory. If it escapes the root prefix directory, reject the request.
 
---


In [ ]:
import os
from pathlib import Path

# Setup a sandboxed upload directory for the lab
sandbox_dir = os.path.abspath("sandbox_uploads")
os.makedirs(sandbox_dir, exist_ok=True)

# Create a demo user file inside sandbox
with open(os.path.join(sandbox_dir, "user1.txt"), "w") as f:
    f.write("User 1 profile text data.")

# 1. Vulnerable File Reader
def vulnerable_read_file(user_filename):
    """Vulnerable path builder using simple concatenation."""
    # DANGEROUS: Concatenation allows escaping
    filepath = os.path.join(sandbox_dir, user_filename)
    print(f"[VULNERABLE OPENING PATH]: {filepath}")
    
    try:
        with open(filepath, "r") as f:
            return f.read()
    except Exception as e:
        return f"Error: {e}"

print("--- Vulnerable Path Resolution (Normal Input) ---")
print(vulnerable_read_file("user1.txt"))



In [ ]:
print("\n--- Directory Traversal Attack ---")
# Attacker tries to escape the sandbox directory
# If this was a web server, they could access system logs or private files.
attack_path = "../../../../../etc/passwd"  # Escapes sandbox_uploads
result = vulnerable_read_file(attack_path)
print(f"Content read: {repr(result[:50])}...")  # Displays start of target file



In [ ]:
# 2. Secure File Reader using resolve() and prefix validation
def secure_read_file(user_filename, base_directory):
    """Secure path builder checking canonical path prefixes."""
    # Convert base directory to absolute path
    base_path = Path(base_directory).resolve()
    
    # Construct combined path
    target_path = Path(base_directory).joinpath(user_filename)
    
    try:
        # Resolve path to canonical absolute representation (removes ../ and symlinks)
        resolved_path = target_path.resolve()
        print(f"[SECURE RESOLVED PATH]: {resolved_path}")
        
        # Verify resolved path starts with the base path prefix
        # is_relative_to is available in Python 3.9+
        if not resolved_path.is_relative_to(base_path):
            return "Security Alert: Access Denied! Path traversal attempt blocked."
            
        with open(resolved_path, "r") as f:
            return f.read()
    except Exception as e:
        return f"Error: {e}"

print("\n--- Secure Path Resolution ---")
print("Reading user1.txt:")
print(secure_read_file("user1.txt", sandbox_dir))

print("\nReading traversal path:")
print(secure_read_file(attack_path, sandbox_dir))
# Expected: Blocks path traversal attempt!



In [ ]:
# Clean up sandbox
if os.path.exists(os.path.join(sandbox_dir, "user1.txt")):
    os.remove(os.path.join(sandbox_dir, "user1.txt"))
if os.path.exists(sandbox_dir):
    os.rmdir(sandbox_dir)
